In [1]:
#0. 最初要改变的变量
Topology_Version = 'gridx'
P=18
N=36

In [2]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os




os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())





backend (before pyplot): QtAgg
backend (after pyplot): qtagg


In [3]:
from pathlib import Path
from src.viz.pyqt_main2 import SatelliteViewer
DATA_DIR = Path(r"D:\paper3")
BASEDIR =  DATA_DIR / "data"


Topology_DIR = 'topology_design'
Topology_Version = 'gridx'
TOPO_CFG_PATH=BASEDIR / Topology_DIR / Topology_Version/"config" / "motif.json"

xml_file = BASEDIR / 'satellitesposition' / "station_visible_satellites_20250106.xml"

In [5]:
import src.io.read_csv as read_csv

analysis_link_dir =BASEDIR / Topology_DIR / Topology_Version/"analysis_link"
df = read_csv.read_csv_generic(analysis_link_dir/"all_pair_global_stat_parallel.csv")
print(df.columns.tolist())
print(df.head())


['PAIR_CSV_NAME', 'count', 'mean', 'median', 'std', 'min', 'p05', 'p10', 'p90', 'p95', 'max', 'time_ratio_rel_lt_0.95', 'time_ratio_rel_lt_0.98', 'time_ratio_rel_ge_0.99']
                              PAIR_CSV_NAME    count      mean    median  \
0  region1--station0-region2--station10.csv  86165.0  0.925244  0.931133   
1  region1--station0-region2--station11.csv  86165.0  0.915844  0.923710   
2  region1--station0-region2--station12.csv  86165.0  0.920518  0.923710   
3  region1--station0-region2--station13.csv  86165.0  0.917543  0.923710   
4  region1--station0-region2--station14.csv  86165.0  0.915561  0.915389   

        std       min       p05       p10       p90       p95       max  \
0  0.015254  0.891762  0.898070  0.906235  0.940539  0.940539  0.940539   
1  0.021103  0.873142  0.881962  0.886385  0.943409  0.943409  0.959679   
2  0.023329  0.860058  0.868746  0.876644  0.943409  0.951985  0.976230   
3  0.022579  0.868746  0.877521  0.886385  0.944353  0.944353  0.960640

In [7]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1) 读取 station_LLA 目录
# 假设：
# - 文件名里能提取出 station 编号，例如 station1.txt / S1.csv / 1.txt
# - 文件内容要么有列名 lat/lon，要么前两列就是 lat/lon
# ============================================================
def _parse_station_id_from_filename(file_path: Path) -> int:
    stem = file_path.stem
    nums = re.findall(r"\d+", stem)
    if not nums:
        raise ValueError(f"文件名里没找到 station id: {file_path.name}")
    return int(nums[0])


def _read_one_station_lla(file_path: Path) -> tuple[float, float]:
    # 先尝试按“有表头”读取
    try:
        df0 = pd.read_csv(file_path, sep=None, engine="python")
        cols_lower = {str(c).strip().lower(): c for c in df0.columns}

        lat_col = None
        lon_col = None

        for k in ["lat", "latitude"]:
            if k in cols_lower:
                lat_col = cols_lower[k]
                break

        for k in ["lon", "longitude", "lng"]:
            if k in cols_lower:
                lon_col = cols_lower[k]
                break

        if lat_col is not None and lon_col is not None and len(df0) > 0:
            lat = float(df0.iloc[0][lat_col])
            lon = float(df0.iloc[0][lon_col])
            return lat, lon
    except Exception:
        pass

    # 否则按“无表头，前两列就是 lat/lon”读取
    df1 = pd.read_csv(file_path, sep=None, engine="python", header=None)
    if df1.shape[1] < 2:
        raise ValueError(f"文件列数不足，无法读取 lat/lon: {file_path}")

    lat = float(df1.iloc[0, 0])
    lon = float(df1.iloc[0, 1])
    return lat, lon


def load_station_lla_dir(station_lla_dir: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(station_lla_dir.iterdir()):
        if not p.is_file():
            continue
        if p.suffix.lower() not in [".csv", ".txt", ".tsv"]:
            continue

        station_id = _parse_station_id_from_filename(p)
        lat, lon = _read_one_station_lla(p)

        rows.append({
            "station_id": station_id,
            "lat": lat,
            "lon": lon,
            "file_name": p.name,
        })

    out = pd.DataFrame(rows).sort_values("station_id").reset_index(drop=True)
    if out.empty:
        raise ValueError(f"station_LLA 目录下没有读到有效文件: {station_lla_dir}")
    return out


# ============================================================
# 2) 从 PAIR_CSV_NAME 解析两个 station id
# 例如：
# region1--station2-region2--station8.csv
# -> (2, 8)
# ============================================================
def parse_station_pair_from_name(pair_csv_name: str) -> tuple[int, int]:
    nums = re.findall(r"station(\d+)", str(pair_csv_name))
    if len(nums) < 2:
        raise ValueError(f"无法从 PAIR_CSV_NAME 解析两个 station: {pair_csv_name}")
    return int(nums[0]), int(nums[1])


# ============================================================
# 3) 构造 station1 视角的数据表
# 输出包含：
# - anchor_station
# - peer_station
# - anchor_lat / anchor_lon
# - peer_lat / peer_lon
# - delta_lat / delta_lon
# - mean
# ============================================================
def build_anchor_pair_table(
    df_global: pd.DataFrame,
    station_lla_df: pd.DataFrame,
    anchor_station: int = 1,
    pair_name_col: str = "PAIR_CSV_NAME",
    value_col: str = "mean",
) -> pd.DataFrame:
    if pair_name_col not in df_global.columns:
        raise ValueError(f"df_global 缺少列: {pair_name_col}")
    if value_col not in df_global.columns:
        raise ValueError(f"df_global 缺少列: {value_col}")

    lla_map = {
        int(row.station_id): (float(row.lat), float(row.lon))
        for row in station_lla_df.itertuples(index=False)
    }

    if anchor_station not in lla_map:
        raise ValueError(f"station_LLA 中找不到 anchor_station={anchor_station}")

    anchor_lat, anchor_lon = lla_map[anchor_station]

    rows = []
    for row in df_global.itertuples(index=False):
        pair_name = getattr(row, pair_name_col)
        stat_value = float(getattr(row, value_col))

        s1, s2 = parse_station_pair_from_name(pair_name)

        if s1 != anchor_station and s2 != anchor_station:
            continue

        peer_station = s2 if s1 == anchor_station else s1
        if peer_station not in lla_map:
            continue

        peer_lat, peer_lon = lla_map[peer_station]

        rows.append({
            "PAIR_CSV_NAME": pair_name,
            "anchor_station": anchor_station,
            "peer_station": peer_station,
            "anchor_lat": anchor_lat,
            "anchor_lon": anchor_lon,
            "peer_lat": peer_lat,
            "peer_lon": peer_lon,
            "delta_lat": anchor_lat - peer_lat,
            "delta_lon": anchor_lon - peer_lon,
            value_col: stat_value,
        })

    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError(f"没有找到和 station{anchor_station} 相关的 pair")
    return out.sort_values("peer_station").reset_index(drop=True)


# ============================================================
# 4) 按经纬度差值分箱，做热力图矩阵
# lat_bin_deg / lon_bin_deg 控制热力图粒度
# ============================================================
def build_delta_heatmap_table(
    df_anchor: pd.DataFrame,
    value_col: str = "mean",
    lat_bin_deg: float = 10.0,
    lon_bin_deg: float = 10.0,
):
    out = df_anchor.copy()

    out["delta_lat_bin"] = np.round(out["delta_lat"] / lat_bin_deg) * lat_bin_deg
    out["delta_lon_bin"] = np.round(out["delta_lon"] / lon_bin_deg) * lon_bin_deg

    grouped = (
        out.groupby(["delta_lat_bin", "delta_lon_bin"], as_index=False)[value_col]
        .mean()
    )

    pivot = grouped.pivot(
        index="delta_lat_bin",
        columns="delta_lon_bin",
        values=value_col,
    )

    pivot = pivot.sort_index(ascending=True)
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)

    return out, grouped, pivot


# ============================================================
# 5) 画热力图
# ============================================================
def plot_delta_heatmap(
    pivot: pd.DataFrame,
    anchor_station: int = 1,
    value_col: str = "mean",
    cmap: str = "viridis",
    figsize=(10, 6),
):
    fig, ax = plt.subplots(figsize=figsize)

    data = pivot.values.astype(float)
    im = ax.imshow(data, origin="lower", aspect="auto", cmap=cmap)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{x:.0f}" for x in pivot.columns], rotation=45, ha="right")

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{y:.0f}" for y in pivot.index])

    ax.set_xlabel("Delta Lon = station1_lon - peer_lon (deg)")
    ax.set_ylabel("Delta Lat = station1_lat - peer_lat (deg)")
    ax.set_title(f"Station {anchor_station} Pair Mean Reliability Heatmap")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(value_col)

    plt.tight_layout()
    return fig, ax


# ============================================================
# 6) 直接运行
# 你前面已经有：
# df = read_csv.read_csv_generic(analysis_link_dir / "all_pair_global_stat_parallel.csv")
# ============================================================
station_lla_dir = BASEDIR / "station_LLA"

station_lla_df = load_station_lla_dir(station_lla_dir)
df_station1 = build_anchor_pair_table(
    df_global=df,
    station_lla_df=station_lla_df,
    anchor_station=1,
    pair_name_col="PAIR_CSV_NAME",
    value_col="mean",
)





df_station1_bin, df_station1_grouped, pivot_station1 = build_delta_heatmap_table(
    df_anchor=df_station1,
    value_col="mean",
    lat_bin_deg=10.0,
    lon_bin_deg=10.0,
)

print(df_station1.head())
print(df_station1_grouped.head())

fig, ax = plot_delta_heatmap(
    pivot_station1,
    anchor_station=1,
    value_col="mean",
    cmap="viridis",
    figsize=(10, 6),
)
plt.show()


                             PAIR_CSV_NAME  anchor_station  peer_station  \
0  region1--station1-region2--station5.csv               1             5   
1  region1--station1-region2--station6.csv               1             6   
2  region1--station1-region2--station7.csv               1             7   
3  region1--station1-region2--station8.csv               1             8   
4  region1--station1-region2--station9.csv               1             9   

   anchor_lat  anchor_lon  peer_lat  peer_lon  delta_lat  delta_lon      mean  
0    -15.7939    -47.8828   12.1140  -86.2362   -27.9079    38.3534  0.917500  
1    -15.7939    -47.8828  -25.7479   28.2293     9.9540   -76.1121  0.923945  
2    -15.7939    -47.8828   30.0444   31.2357   -45.8383   -79.1185  0.925365  
3    -15.7939    -47.8828   36.7538    3.0588   -52.5477   -50.9416  0.918521  
4    -15.7939    -47.8828    9.0320   38.7469   -24.8259   -86.6297  0.916307  
   delta_lat_bin  delta_lon_bin      mean
0          -70.0     

In [12]:
import numpy as np
import matplotlib.pyplot as plt


def wrap_lon_diff(delta_lon):
    """
    把经度差规整到 [-180, 180]
    例如:
      190  -> -170
      -200 -> 160
    """
    return ((delta_lon + 180) % 360) - 180


def plot_station_delta_scatter(
    df_anchor,
    value_col="mean",
    annotate=False,
    xlim=(-180, 180),
    ylim=(-180, 180),
    vmin=0.0,
    vmax=1.0,
):
    df_plot = df_anchor.copy()

    # 关键：经度差先规整到 [-180, 180]
    df_plot["delta_lon_plot"] = df_plot["delta_lon"].apply(wrap_lon_diff)

    fig, ax = plt.subplots(figsize=(8, 8))

    sc = ax.scatter(
        df_plot["delta_lon_plot"],
        df_plot["delta_lat"],
        c=df_plot[value_col],
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        s=90,
        edgecolors="black",
        linewidths=0.5,
    )

    # 固定坐标范围
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    # 固定坐标比例，避免横纵拉伸
    ax.set_aspect("equal", adjustable="box")

    # 固定刻度，方便肉眼比较
    ax.set_xticks(np.arange(-180, 181, 30))
    ax.set_yticks(np.arange(-180, 181, 30))

    ax.set_xlabel("Delta Lon (deg)")
    ax.set_ylabel("Delta Lat (deg)")
    ax.set_title("Pair Mean Reliability Scatter")
    ax.grid(alpha=0.3, linestyle="--")

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(value_col)

    if annotate:
        for row in df_plot.itertuples(index=False):
            ax.text(
                row.delta_lon_plot,
                row.delta_lat,
                f"{row.peer_station}",
                fontsize=8,
                ha="left",
                va="bottom",
            )

    plt.tight_layout()
    plt.show(block=False)
    plt.pause(0.01)

    return fig, ax


In [16]:
GLOBAL_VMIN = 0.915419632740149
GLOBAL_VMAX = 0.9985717518714096


In [19]:
len(df_station1)

26

In [17]:
fig, ax = plot_station_delta_scatter(
    df_station1,
    value_col="mean",
    annotate=True,
    xlim=(-180, 180),
    ylim=(-180, 180),
    vmin=GLOBAL_VMIN,
    vmax=GLOBAL_VMAX,
)


In [14]:
print(df["mean"].describe())
print(df["mean"].min(), df["mean"].max())


count    340.000000
mean       0.936213
std        0.017057
min        0.915420
25%        0.925178
50%        0.930436
75%        0.942081
max        0.998572
Name: mean, dtype: float64
0.915419632740149 0.9985717518714096


In [20]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


def wrap_lon_diff(delta_lon):
    """
    把经度差规整到 [-180, 180]
    """
    return ((delta_lon + 180) % 360) - 180


def build_global_color_norm(df_all, value_col="mean", gamma=0.5):
    """
    用总表 df_all 一次性构造全局统一色标。
    后续所有 station 的图都复用这个 norm。
    """
    vmin = float(df_all[value_col].min())
    vmax = float(df_all[value_col].max())

    norm = mcolors.PowerNorm(
        gamma=gamma,
        vmin=vmin,
        vmax=vmax,
    )
    return norm, vmin, vmax


def plot_station_delta_scatter(
    df_anchor,
    *,
    anchor_station,
    value_col="mean",
    norm=None,
    cmap="turbo",
    annotate=True,
    figsize=(8, 8),
    xlim=(-180, 180),
    ylim=(-180, 180),
    xtick_step=30,
    ytick_step=30,
    point_size=110,
    edge_linewidth=0.25,
    title=None,
):
    """
    df_anchor 需要至少包含：
      delta_lon, delta_lat, peer_station, mean
    """

    df_plot = df_anchor.copy()
    df_plot["delta_lon_plot"] = df_plot["delta_lon"].apply(wrap_lon_diff)

    fig, ax = plt.subplots(figsize=figsize)

    sc = ax.scatter(
        df_plot["delta_lon_plot"],
        df_plot["delta_lat"],
        c=df_plot[value_col],
        cmap=cmap,
        norm=norm,
        s=point_size,
        edgecolors="black",
        linewidths=edge_linewidth,
        alpha=0.95,
    )

    if annotate:
        for row in df_plot.itertuples(index=False):
            ax.text(
                row.delta_lon_plot,
                row.delta_lat,
                f"{row.peer_station}",
                fontsize=8,
                ha="left",
                va="bottom",
            )

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xticks(np.arange(xlim[0], xlim[1] + 1, xtick_step))
    ax.set_yticks(np.arange(ylim[0], ylim[1] + 1, ytick_step))

    ax.set_xlabel("Delta Lon = anchor_lon - peer_lon (deg)")
    ax.set_ylabel("Delta Lat = anchor_lat - peer_lat (deg)")

    if title is None:
        title = f"Station {anchor_station} Pair Mean Reliability"
    ax.set_title(title)

    ax.grid(alpha=0.3, linestyle="--")

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(value_col)

    plt.tight_layout()
    plt.show(block=False)
    plt.pause(0.01)

    return fig, ax


In [22]:
# 用总表 df 一次性生成全局统一色标
GLOBAL_NORM, GLOBAL_VMIN, GLOBAL_VMAX = build_global_color_norm(
    df,
    value_col="mean",
    gamma=0.5,
)

print("GLOBAL_VMIN =", GLOBAL_VMIN)
print("GLOBAL_VMAX =", GLOBAL_VMAX)


GLOBAL_VMIN = 0.915419632740149
GLOBAL_VMAX = 0.9985717518714096


In [33]:
fig, ax = plot_station_delta_scatter(
    df_station[1],
    anchor_station=1,
    value_col="mean",
    norm=GLOBAL_NORM,
    cmap="turbo",
    annotate=True,
)


In [25]:
def extract_all_station_ids_from_global_df(df_global, pair_name_col="PAIR_CSV_NAME"):
    station_ids = set()

    for pair_name in df_global[pair_name_col]:
        s1, s2 = parse_station_pair_from_name(pair_name)
        station_ids.add(int(s1))
        station_ids.add(int(s2))

    return sorted(station_ids)


def build_all_anchor_pair_tables(
    df_global,
    station_lla_df,
    pair_name_col="PAIR_CSV_NAME",
    value_col="mean",
    station_ids=None,
):
    """
    返回两个结果：

    1) df_station_list
       按 station_ids 的顺序存放
       例如如果 station_ids = [1,2,3,...]
       那么 df_station_list[0] 就是 station1
       df_station_list[1] 就是 station2

    2) df_station_map
       用真实 station_id 直接索引
       例如 df_station_map[1] 就是 station1
    """
    if station_ids is None:
        station_ids = extract_all_station_ids_from_global_df(
            df_global,
            pair_name_col=pair_name_col,
        )

    df_station_list = []
    df_station_map = {}

    for station_id in station_ids:
        df_anchor = build_anchor_pair_table(
            df_global=df_global,
            station_lla_df=station_lla_df,
            anchor_station=station_id,
            pair_name_col=pair_name_col,
            value_col=value_col,
        )
        df_station_list.append(df_anchor)
        df_station_map[station_id] = df_anchor

    return df_station_list, df_station_map, station_ids


In [29]:

station_ids = [sid for sid in extract_all_station_ids_from_global_df(df) if sid >= 1]

df_station, df_station_map, station_ids = build_all_anchor_pair_tables(
    df_global=df,
    station_lla_df=station_lla_df,
    pair_name_col="PAIR_CSV_NAME",
    value_col="mean",
    station_ids=station_ids,
)


In [34]:
## export

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


def wrap_lon_diff(delta_lon):
    return ((delta_lon + 180) % 360) - 180


def build_global_color_norm(df_all, value_col="mean", gamma=0.5):
    vmin = float(df_all[value_col].min())
    vmax = float(df_all[value_col].max())

    norm = mcolors.PowerNorm(
        gamma=gamma,
        vmin=vmin,
        vmax=vmax,
    )
    return norm, vmin, vmax


def plot_station_delta_scatter(
    df_anchor,
    *,
    anchor_station,
    value_col="mean",
    norm=None,
    cmap="turbo",
    annotate=True,
    figsize=(8, 8),
    xlim=(-180, 180),
    ylim=(-180, 180),
    xtick_step=30,
    ytick_step=30,
    point_size=110,
    edge_linewidth=0.25,
    title=None,
    show=True,
):
    df_plot = df_anchor.copy()
    df_plot["delta_lon_plot"] = df_plot["delta_lon"].apply(wrap_lon_diff)

    fig, ax = plt.subplots(figsize=figsize)

    sc = ax.scatter(
        df_plot["delta_lon_plot"],
        df_plot["delta_lat"],
        c=df_plot[value_col],
        cmap=cmap,
        norm=norm,
        s=point_size,
        edgecolors="black",
        linewidths=edge_linewidth,
        alpha=0.95,
    )

    if annotate:
        for row in df_plot.itertuples(index=False):
            ax.text(
                row.delta_lon_plot,
                row.delta_lat,
                f"{row.peer_station}",
                fontsize=8,
                ha="left",
                va="bottom",
            )

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xticks(np.arange(xlim[0], xlim[1] + 1, xtick_step))
    ax.set_yticks(np.arange(ylim[0], ylim[1] + 1, ytick_step))

    ax.set_xlabel("Delta Lon = anchor_lon - peer_lon (deg)")
    ax.set_ylabel("Delta Lat = anchor_lat - peer_lat (deg)")

    if title is None:
        title = f"Station {anchor_station} Pair Mean Reliability"
    ax.set_title(title)

    ax.grid(alpha=0.3, linestyle="--")

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(value_col)

    plt.tight_layout()

    if show:
        plt.show(block=False)
        plt.pause(0.01)

    return fig, ax



In [36]:
OUT_DIR =analysis_link_dir/"jpg"


if not OUT_DIR.exists():
    OUT_DIR.mkdir(parents=True, exist_ok=True)



In [37]:
for station_id, df_one in zip(station_ids, df_station):
    fig, ax = plot_station_delta_scatter(
        df_one,
        anchor_station=station_id,
        value_col="mean",
        norm=GLOBAL_NORM,
        cmap="turbo",
        annotate=True,
        show=False,
    )

    out_path = OUT_DIR / f"station{station_id}_delta_scatter.jpg"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"saved: {out_path}")


saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station1_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station2_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station3_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station4_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station5_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station6_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station7_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station8_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station9_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station10_delta_scatter.jpg
saved: D:\paper3\data\topology_design\gridx\analysis_link\jpg\station11_delta_scatter.jpg
saved: D:\paper3\da

# 数据处理
这一章节，我们要开始数据后处理，就是上面数据导出后，我们需要进行同轨异轨的一个分析验证


In [3]:
# from pathlib import Path
# import re
# import pandas as pd
#
#
# def read_pair_csv(
#     csv_path,
#     *,
#     time_col="time",
#     station_a_col="station_a",
#     station_b_col="station_b",
#     path_col="path",
#     ensure_int=True,
# ):
#     """
#     读取一份 station 对的最短路径 CSV。
#
#     支持两种情况：
#     1) CSV 本身包含 station_a / station_b 列
#     2) CSV 没有这两列时，从文件名中提取（如 region1--station12-region2--station19.csv）
#     """
#     path = Path(csv_path)
#     df = pd.read_csv(path)
#
#     # 缺列校验
#     if time_col not in df.columns:
#         raise ValueError(f"CSV 缺少列: {time_col} -> {path}")
#
#     # 文件名兜底解析 station 对
#     if station_a_col not in df.columns or station_b_col not in df.columns:
#         m = re.match(r".*station(\d+)-[^-]*-station(\d+)", path.stem)
#         if not m:
#             raise ValueError(
#                 f"CSV 缺少 {station_a_col}/{station_b_col}，且文件名无法解析: {path.name}"
#             )
#         df[station_a_col] = int(m.group(1))
#         df[station_b_col] = int(m.group(2))
#
#     # 可选字段保证存在
#     if path_col not in df.columns:
#         df[path_col] = ""
#
#     if ensure_int:
#         df[time_col] = pd.to_numeric(df[time_col], errors="coerce").astype("Int64")
#         df[station_a_col] = pd.to_numeric(df[station_a_col], errors="coerce").astype("Int64")
#         df[station_b_col] = pd.to_numeric(df[station_b_col], errors="coerce").astype("Int64")
#
#     # 简单标准列
#     return (
#         df
#         .loc[:, [time_col, station_a_col, station_b_col, path_col] + [
#             c for c in df.columns
#             if c not in {time_col, station_a_col, station_b_col, path_col}
#         ]]
#         .copy()
#     )


In [6]:


DATA_DIR = Path(r"D:\paper3")
BASEDIR =  DATA_DIR / "data"

Topology_DIR = 'topology_design'

In [7]:
FIGURE_DIR = BASEDIR / Topology_DIR / Topology_Version/"path"

data_DIR = Path(FIGURE_DIR) / "region_pairs_0_86164"   # 你导出的原始 csv 目录


In [8]:
import  src.paper3_postprocess.read_path_csv as read_path_csv
# data_DIR = Path(FIGURE_DIR) / "region1_to_region2_0_100"   # 你导出的原始 csv 目录

df = read_path_csv.read_pair_csv(data_DIR / "region1--station2-region2--station8.csv")


In [9]:
# 1) 从 df 里拿当前 station pair
df_s6_s8 = df
station_a = int(df_s6_s8["station_a"].iloc[0])
station_b = int(df_s6_s8["station_b"].iloc[0])


,time,station_a,station_b,path,best_s,best_d,path_indexed
0,0,0,19,86->85->120->119->154->153->188->187->222->221...,86,362,"1:86,2:85,3:120,4:119,5:154,6:153,7:188,8:187,..."
1,1,0,19,86->85->120->119->154->153->188->187->222->221...,86,362,"1:86,2:85,3:120,4:119,5:154,6:153,7:188,8:187,..."
2,2,0,19,86->85->120->119->154->153->188->187->222->221...,86,362,"1:86,2:85,3:120,4:119,5:154,6:153,7:188,8:187,..."
3,3,0,19,86->85->120->119->154->153->188->187->222->221...,86,362,"1:86,2:85,3:120,4:119,5:154,6:153,7:188,8:187,..."
4,4,0,19,86->85->120->119->154->153->188->187->222->221...,86,362,"1:86,2:85,3:120,4:119,5:154,6:153,7:188,8:187,..."
...,...,...,...,...,...,...,...
96,96,0,19,85->120->119->154->153->188->187->222->221->25...,85,361,"1:85,2:120,3:119,4:154,5:153,6:188,7:187,8:222..."
97,97,0,19,85->120->119->154->153->188->187->222->221->25...,85,361,"1:85,2:120,3:119,4:154,5:153,6:188,7:187,8:222..."
98,98,0,19,85->120->119->154->153->188->187->222->221->25...,85,361,"1:85,2:120,3:119,4:154,5:153,6:188,7:187,8:222..."
99,99,0,19,85->120->119->154->153->188->187->222->221->25...,85,361,"1:85,2:120,3:119,4:154,5:153,6:188,7:187,8:222..."


In [7]:
dfpath = df["path"]

In [10]:
# ========== 路径 → intra/inter 链路分类 ==========
# 星座参数
# P = 18
# N = 36

import src.model.get_intra_inter_link as get_intra_inter_link


all_intra = []
all_inter = []

for idx, path_str in enumerate(df["path"]):
    intra, inter = get_intra_inter_link.parse_path_links(path_str, N=N)
    all_intra.append(intra)
    all_inter.append(inter)

# 写回 DataFrame
df["intra_links"] = all_intra    # 每行是 [(src,dst), ...] 的 list
df["inter_links"] = all_inter

# 同时统计跳数
df["intra_hops"] = df["intra_links"].apply(len)
df["inter_hops"] = df["inter_links"].apply(len)
df["total_hops"] = df["intra_hops"] + df["inter_hops"]


In [12]:
df["intra_hops"]


0      6
1      6
2      6
3      6
4      6
      ..
96     6
97     6
98     6
99     6
100    6
Name: intra_hops, Length: 101, dtype: int64

In [11]:
def plot_intra_inter_hops_over_time(
    df,
    *,
    time_col="time",
    intra_col="intra_hops",
    inter_col="inter_hops",
    figsize=(10, 4),
    title="Intra-orbit vs Inter-orbit hops over time",
    show=True,
    save=None,
    save_dir="figs",
    basename="intra_inter_hops",
    formats=("png", "pdf"),
    dpi=300,
    return_handles=True,
):
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    base_rc = {
        "font.family": "Times New Roman",
        "font.size": 14,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "axes.linewidth": 1.2,
    }

    plt.ion()
    with mpl.rc_context(base_rc):
        fig, ax = plt.subplots(figsize=figsize)

        t = df[time_col]

        ax.plot(t, df[intra_col], marker='o', markersize=3,
                linewidth=1.5, color='#2196F3', label='Intra-orbit hops')
        ax.plot(t, df[inter_col], marker='s', markersize=3,
                linewidth=1.5, color='#F44336', label='Inter-orbit hops')
        ax.plot(t, df[intra_col] + df[inter_col], marker='^', markersize=3,
                linewidth=1.2, color='#888888', linestyle='--', label='Total hops')

        ax.set_xlabel("Time Step")
        ax.set_ylabel("Hops")
        ax.set_title(title)
        ax.legend(framealpha=0.9)
        ax.grid(alpha=0.3, linestyle="--")
        plt.tight_layout()

        if show:
            plt.show(block=False)
            try:
                plt.pause(0.01)
            except Exception:
                pass

    if save:
        from pathlib import Path
        targets = []
        if save is True:
            outdir = Path(save_dir); outdir.mkdir(parents=True, exist_ok=True)
            for ext in formats:
                targets.append(outdir / f"{basename}.{ext.lstrip('.')}")
        else:
            targets = [save] if isinstance(save, (str, Path)) else list(save)
        for p in targets:
            p = Path(p)
            p.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(p, dpi=dpi, bbox_inches="tight")

    return (fig, ax, df) if return_handles else None


In [12]:
plot_intra_inter_hops_over_time(
    df,
    title="Station A ↔ Station B: Intra vs Inter hops (× grid)",
)


(<Figure size 1000x400 with 1 Axes>,
 <Axes: title={'center': 'Station A ↔ Station B: Intra vs Inter hops (× grid)'}, xlabel='Time Step', ylabel='Hops'>,
         time  station_a  station_b  \
 0          0          2          8   
 1          1          2          8   
 2          2          2          8   
 3          3          2          8   
 4          4          2          8   
 ...      ...        ...        ...   
 86160  86160          2          8   
 86161  86161          2          8   
 86162  86162          2          8   
 86163  86163          2          8   
 86164  86164          2          8   
 
                                                     path  best_s  best_d  \
 0      179->214->213->248->247->282->319->354->391->4...     179     535   
 1      179->214->213->248->247->282->319->354->391->4...     179     535   
 2      179->214->213->248->247->282->319->354->391->4...     179     535   
 3      179->214->213->248->247->282->319->354->391->4...     179   

In [19]:
def plot_route_reliability_over_time(
    df,
    *,
    reliability_col,
    time_col="time",
    figsize=(10, 4),
    title=None,
    show=True,
    save=None,
    save_dir="figs",
    basename="route_reliability",
    formats=("png", "pdf"),
    dpi=300,
    return_handles=True,
):
    """
    只负责绘图。
    要求 df 中已经有 reliability_col 这一列。
    """
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from pathlib import Path

    reliability = df[reliability_col].astype(float)

    base_rc = {
        "font.family": "Times New Roman",
        "font.size": 14,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "axes.linewidth": 1.2,
    }

    plt.ion()
    with mpl.rc_context(base_rc):
        fig, ax = plt.subplots(figsize=figsize)

        ax.plot(
            df[time_col],
            reliability * 100,
            marker="o",
            markersize=3,
            linewidth=1.5,
            color="#4CAF50",
        )

        ax.set_xlabel("Time Step")
        ax.set_ylabel("Route Reliability (%)")
        ax.set_ylim(
            bottom=max(0, (reliability.min() * 100) - 2),
            top=min(100, (reliability.max() * 100) + 1),
        )

        if title is None:
            title = reliability_col
        ax.set_title(title)
        ax.grid(alpha=0.3, linestyle="--")
        plt.tight_layout()

        if show:
            plt.show(block=False)
            try:
                plt.pause(0.01)
            except Exception:
                pass

    if save:
        targets = []
        if save is True:
            outdir = Path(save_dir)
            outdir.mkdir(parents=True, exist_ok=True)
            for ext in formats:
                targets.append(outdir / f"{basename}.{ext.lstrip('.')}")
        else:
            targets = [save] if isinstance(save, (str, Path)) else list(save)

        for p in targets:
            p = Path(p)
            p.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(p, dpi=dpi, bbox_inches="tight")

    return (fig, ax) if return_handles else None




In [17]:
import  src.paper3_postprocess.route_reliable as route_reliable

In [20]:
rel_0999_099 = route_reliable.compute_route_reliability_series(
    df,
    p_intra=0.999,
    p_inter=0.99,
)

df[rel_0999_099.name] = rel_0999_099

fig, ax = plot_route_reliability_over_time(
    df,
    reliability_col=rel_0999_099.name,
    title="End-to-End Route Reliability",
    show=True,
)


In [24]:
df = df.copy()
df["rel_0999_099"] = rel_0999_099   # rel_0999_099 是你已算好的 Series


In [13]:
# ALL_EDGES_WINDOW = {step: STATIC_EDGES for step in range(WIN_START, WIN_END + 1)}


In [13]:
import sys
from src.config.viewer_config import ViewerConfig
import src.io.operate_group_data as operate_group_data

# station_a = int(df_s6_s8["station_a"].iloc[0])
# station_b = int(df_s6_s8["station_b"].iloc[0])
#
S6 = series_by_station[station_a]
S8 = series_by_station[station_b]

group_data_s6_s8 = operate_group_data.build_stationpair_group_data(
    S6, S8, WIN_START, WIN_END
)

PAIR_CFG = ViewerConfig(
    name=f"S{station_a}_S{station_b}_VERIFY",
    N=cfg.N,
    P=cfg.P,
    station_groups={
        0: {"name": f"S{station_a}", "stations": [station_a]},
        1: {"name": f"S{station_b}", "stations": [station_b]},
    },
    group_colors=["#ff4d4f", "#2f54eb"],
)

path_by_step = {}
for row in df_s6_s8.itertuples(index=False):
    p = getattr(row, "path", "")
    if isinstance(p, str) and p.strip():
        path_by_step[int(row.time)] = [int(x) for x in p.split("->")]



In [ ]:
# viewer = SatelliteViewer(group_data_s6_s8, PAIR_CFG)
# viewer.setWindowTitle(f"verify station{station_a}-station{station_b}")
# viewer.resize(1400, 800)
#
# # 不再传 step->adj 的大字典
# viewer.static_edges = STATIC_EDGES
# viewer.static_topology = True
#
# viewer.set_paths(path_by_step)
# viewer.show()
#
# if not hasattr(sys.modules[__name__], "_viewer_list"):
#     _viewer_list = []
# _viewer_list.append(viewer)


In [41]:
import time

t0 = time.perf_counter()
group_data_s6_s8 = operate_group_data.build_stationpair_group_data(
    S6, S8, WIN_START, WIN_END
)
print("build_stationpair_group_data:", time.perf_counter() - t0)

t0 = time.perf_counter()
path_by_step = {}
for row in df_s6_s8.itertuples(index=False):
    p = getattr(row, "path", "")
    if isinstance(p, str) and p.strip():
        path_by_step[int(row.time)] = [int(x) for x in p.split("->")]
print("build path_by_step:", time.perf_counter() - t0)

t0 = time.perf_counter()
viewer = SatelliteViewer(group_data_s6_s8, PAIR_CFG)
viewer.setWindowTitle(f"verify station{station_a}-station{station_b}")
viewer.resize(1400, 800)
viewer.static_edges = STATIC_EDGES
viewer.static_topology = True
viewer.set_paths(path_by_step)
viewer.show()
print("viewer part:", time.perf_counter() - t0)


build_stationpair_group_data: 1.6662352997809649
build path_by_step: 0.6177091002464294
viewer part: 0.3936364999972284


In [30]:
df.loc[df['time'] == 29651, ['time', 'station_a', 'station_b', 'path']]

,time,station_a,station_b,path
29651,29651,0,19,616->581->582->547->548->513->514->479->480->5...


In [29]:
import numpy as np

max_idx = np.argmin(rel_0999_099)
max_val = rel_0999_099[max_idx]

print("最大值下标:", max_idx)
print("最大值:", max_val)

最大值下标: 29561
最大值: 0.985104546362002


In [21]:
import numpy as np
import pandas as pd

def reliability_global_stats(df, rel_col="rel_0999_099"):
    x = pd.to_numeric(df[rel_col], errors="coerce").dropna()
    return pd.Series({
        "count": int(x.size),
        "mean": float(x.mean()),
        "median": float(x.median()),
        "std": float(x.std(ddof=1)),
        "min": float(x.min()),
        "p05": float(x.quantile(0.05)),
        "p10": float(x.quantile(0.10)),
        "p90": float(x.quantile(0.90)),
        "p95": float(x.quantile(0.95)),
        "max": float(x.max()),
        "time_ratio_rel_lt_0.95": float((x < 0.95).mean()),
        "time_ratio_rel_lt_0.98": float((x < 0.98).mean()),
        "time_ratio_rel_ge_0.99": float((x >= 0.99).mean()),
    }, name=rel_col)

def reliability_window_stats(df, rel_col="rel_0999_099", time_col="time", window_sec=300):
    d = df[[time_col, rel_col]].copy()
    d[time_col] = d[time_col].astype(int)
    d[rel_col] = pd.to_numeric(d[rel_col], errors="coerce")
    t0 = int(d[time_col].min())
    d["win_id"] = ((d[time_col] - t0) // int(window_sec)).astype(int)
    out = d.groupby("win_id", as_index=False).agg(
        start_time=(time_col, "min"),
        end_time=(time_col, "max"),
        rel_mean=(rel_col, "mean"),
        rel_min=(rel_col, "min"),
        rel_p05=(rel_col, lambda s: s.quantile(0.05)),
        rel_p95=(rel_col, lambda s: s.quantile(0.95)),
    )
    out["duration_sec"] = out["end_time"] - out["start_time"] + 1
    return out

def path_switch_stats(df, path_col="path", time_col="time"):
    d = df[[time_col, path_col]].sort_values(time_col).copy()
    d[path_col] = d[path_col].fillna("")
    d["switch"] = d[path_col].ne(d[path_col].shift(1))
    if len(d) > 0:
        d.iloc[0, d.columns.get_loc("switch")] = False
    d["seg_id"] = d["switch"].cumsum()
    seg = d.groupby("seg_id", as_index=False).agg(
        path=(path_col, "first"),
        start_time=(time_col, "min"),
        end_time=(time_col, "max"),
    )
    seg["duration_sec"] = seg["end_time"] - seg["start_time"] + 1
    summary = pd.Series({
        "path_switch_count": int(d["switch"].sum()),
        "segment_count": int(len(seg)),
        "avg_dwell_sec": float(seg["duration_sec"].mean()) if len(seg) else 0.0,
        "median_dwell_sec": float(seg["duration_sec"].median()) if len(seg) else 0.0,
        "max_dwell_sec": int(seg["duration_sec"].max()) if len(seg) else 0,
    }, name="path_switch_summary")
    return summary, seg


In [25]:
global_stat = reliability_global_stats(df, rel_col="rel_0999_099")
win_5min = reliability_window_stats(df, rel_col="rel_0999_099", window_sec=300)
switch_summary, switch_segments = path_switch_stats(df, path_col="path")

print(global_stat)
print(win_5min.head())
print(switch_summary)


count                     86165.000000
mean                          0.916194
median                        0.915389
std                           0.010397
min                           0.902574
p05                           0.902574
p10                           0.902574
p90                           0.934910
p95                           0.934910
max                           0.935845
time_ratio_rel_lt_0.95        1.000000
time_ratio_rel_lt_0.98        1.000000
time_ratio_rel_ge_0.99        0.000000
Name: rel_0999_099, dtype: float64
   win_id  start_time  end_time  rel_mean   rel_min   rel_p05   rel_p95  \
0       0           0       299  0.902839  0.902574  0.902574  0.903478   
1       1         300       599  0.903202  0.902574  0.902574  0.911691   
2       2         600       899  0.911436  0.910779  0.910779  0.911691   
3       3         900      1199  0.910834  0.903478  0.903478  0.911691   
4       4        1200      1499  0.908426  0.902574  0.902574  0.911691   

   dura